<center>LOCK CLASS

Esempio fatto in aula per mostrare serializzazione "sottobanco" dei thread

while true; do python prova.py; sleep 1; done

In [ ]:
import threading #permette di risolvere il nome della classe

x=10

def increment(val):
    """
        increment incrementerà x rispetto al valore val
        """
    global x #forzo a operare i 2 thread sullo stesso oggetto -> intero è immutabile (creerebbe un nuovo oggetto se passato come parametro)


    local_counter = x
    local_counter += val

    x=local_counter

    print(f'{threading.current_thread().name} incrementa x di {val}, x: {x}') #ritorna 'identificativo' del thread

t1 = threading.Thread(target=increment, args = (5,)) #il paramtro è una tupla, metti il singleton
t2 = threading.Thread(target=increment, args = (10, ))

t1.start()
t2.start()

t1.join()
t2.join()

print(f'{threading.current_thread().name} valore finale di x: {x}')

#il ris finale è sempre 25-> sotto il banco, i thread sono SERIALIZZATI (il codice è SBAAGLIATO, c'è una race condition -> in C non funzionerebbe)

--> provo a aggiungere semplicemente una sleep-> vedo che il risultato non è più corretto a causa della race condition

In [11]:
import threading #permette di risolvere il nome della classe
import time

x=10

def increment(val):
    """
        increment incrementerà x rispetto al valore val
        """
    global x #forzo a operare i 2 thread sullo stesso oggetto -> intero è immutabile (creerebbe un nuovo oggetto se passato come parametro)


    local_counter = x
    local_counter += val

    time.sleep(1) #aggiunta

    x=local_counter

    print(f'{threading.current_thread().name} incrementa x di {val}, x: {x}') #ritorna 'identificativo' del thread

t1 = threading.Thread(target=increment, args = (5,)) #il paramtro è una tupla, metti il singleton
t2 = threading.Thread(target=increment, args = (10, ))

t1.start()
t2.start()

t1.join()
t2.join()

print(f'{threading.current_thread().name} valore finale di x: {x}')
#Ora ho la race condition! A volte esce 20, a volte 15 ma mai il risultato corretto (25)

Thread-24 (increment) incrementa x di 5, x: 15
Thread-25 (increment) incrementa x di 10, x: 20
MainThread valore finale di x: 20


Mettendo la sleep ho reso evidente l'errore! Come risolvo? Creando un oggetto lock

2 stati: {locked, unlocked}

2 primitive: {_l.acquire()_, _l.release()_}

In [12]:
import threading #permette di risolvere il nome della classe
import time
from threading import Lock

x=10

def increment(val, lock):
    """
        increment incrementerà x rispetto al valore val
        """
    global x #forzo a operare i 2 thread sullo stesso oggetto -> intero è immutabile (creerebbe un nuovo oggetto se passato come parametro)

    lock.acquire()

    local_counter = x
    local_counter += val

    time.sleep(1)

    x=local_counter

    print(f'{threading.current_thread().name} incrementa x di {val}, x: {x}') #ritorna 'identificativo' del thread

    lock.release()

l = Lock() #devo passarlo a tutti i thread

t1 = threading.Thread(target=increment, args = (5,l))
t2 = threading.Thread(target=increment, args = (10,l))

t1.start()
t2.start()

t1.join()
t2.join()

print(f'{threading.current_thread().name} valore finale di x: {x}')

Thread-26 (increment) incrementa x di 5, x: 15
Thread-27 (increment) incrementa x di 10, x: 25
MainThread valore finale di x: 25


Esempio di uso del Lock con primitiva with:

In [13]:
from threading import Thread, Lock

counter = 0

def increase(by, lock:Lock):
    global counter
    with lock:

        local_counter = counter
        local_counter += by
        counter = local_counter
        print(f'counter = {counter}')

if __name__ == '__main__':
    lock = Lock()

    t1 = Thread(target=increase, args=(10, lock))
    t2 = Thread(target=increase, args=(20, lock))
    # start the threads
    t1.start()
    t2.start()
    # wait for the threads to complete
    t1.join()
    t2.join()
    print(f'The final counter is {counter}')

counter = 10
counter = 30
The final counter is 30


--- RLOCK CLASS

Oltre ai concetti già visti nei lock, su aggiungono 2 ulteriori: 

{owning thread, recursion level}

In [17]:
import threading

class SharedResource:
    def __init__(self):
        self.lock = threading.RLock()
        #self.lock = threading.Lock() #se provo a mettere lock normale, vado in deadlock
        self.value = 0

    def increment(self):
        with self.lock:
            self.value +=1
            self._log() #metodo che usa un lock

    def _log(self):
        with self.lock: #reentrant locking
            print(f"Value is now {self.value}, con owner: {self.lock._is_owned}")

resource = SharedResource()

def worker():
    for _ in range(5):
        resource.increment()

threads = [threading.Thread(target=worker) for _ in range(2)]

for t in threads:
    t.start()

for t in threads:
    t.join()


Value is now 1, con owner: <built-in method _is_owned of _thread.RLock object at 0x7fd1f15ff2c0>
Value is now 2, con owner: <built-in method _is_owned of _thread.RLock object at 0x7fd1f15ff2c0>
Value is now 3, con owner: <built-in method _is_owned of _thread.RLock object at 0x7fd1f15ff2c0>
Value is now 4, con owner: <built-in method _is_owned of _thread.RLock object at 0x7fd1f15ff2c0>
Value is now 5, con owner: <built-in method _is_owned of _thread.RLock object at 0x7fd1f15ff2c0>
Value is now 6, con owner: <built-in method _is_owned of _thread.RLock object at 0x7fd1f15ff2c0>
Value is now 7, con owner: <built-in method _is_owned of _thread.RLock object at 0x7fd1f15ff2c0>
Value is now 8, con owner: <built-in method _is_owned of _thread.RLock object at 0x7fd1f15ff2c0>
Value is now 9, con owner: <built-in method _is_owned of _thread.RLock object at 0x7fd1f15ff2c0>
Value is now 10, con owner: <built-in method _is_owned of _thread.RLock object at 0x7fd1f15ff2c0>


--- CONDITION CLASS --> sfrutta un lock o Rlock, eventualmente già esistente

Fornisce i metodi di _acquire()_ , _release()_, _wait()_, _wait_for()_, _notify()_, _notify_all()_

In [21]:
#esempio d'uso ??????????
from threading import Condition

cv_lock = threading.Lock()
cv = Condition(lock=cv_lock)

# Consume one item
with cv:
    while not an_item_is_available(queue):
        cv.wait()
        get_an_available_item(queue)
# Produce one item
with cv:
    make_an_item_available(queue)
    cv.notify()

NameError: name 'an_item_is_available' is not defined

--- SEMAPHORE CLASS istanzia un oggetto semaforo dotato di un contatore interno:
1) Inizializzato alla creazione del semaforo
2) Decremetato a ogni _acquire()_
3) Incrementato a ogni _release()_



In [24]:
#esempio d'uso
from threading import *
from time import sleep
from random import random

# creating thread instance where count = 3
sem = Semaphore(3)

def display(name, s:threading.Semaphore):
    s.acquire()

    value = random()
    sleep(value)
    print(f'Thread {name} got {value}')

    s.release()

if __name__ == '__main__':
    threads = []
    # creating and starting multiple threads
    for i in range(10):
        t = Thread(target = display , args = ('Thread-' + str(i),sem))
        threads.append(t)
        t.start()
    # wait for the threads to complete
    for thread in threads:
        thread.join()

Thread Thread-0 got 0.10288893621145179
Thread Thread-1 got 0.35634987883374636
Thread Thread-3 got 0.37523094005685576
Thread Thread-5 got 0.09816128277463776
Thread Thread-2 got 0.8051598673289698
Thread Thread-4 got 0.627132397232119
Thread Thread-7 got 0.3557418005460461
Thread Thread-6 got 0.7864027540697897
Thread Thread-8 got 0.9982497820411085
Thread Thread-9 got 0.9084126945303955


--- EVENT CLASS
Evento è basato su una variabile condition ma con API più snella

Ogni event è caratterizzato da flag booleano

Metodi:
- _set()_
- _clear()_
- _wait()_
- _is_set()_ (test)

In [26]:
import threading as thd
import time

def task(event, timeout):
    print("Started thread but waiting for event...")
    # make the thread wait for event with timeout set
    event_set = event.wait(timeout)
    if event_set:
        print("Event received, releasing thread...")
    else:
        print("Time out, moving ahead without event...")

if __name__ == '__main__':

    # initializing the event object
    e = thd.Event()
    
    # starting the thread
    thread = thd.Thread(name='Event-Thread', target=task, args=(e,4))
    thread.start()
    
    # sleeping the main thread for 3 seconds
    time.sleep(3)
    
    # generating the event
    e.set()
    print("Event is set.")
    
    # wait for the thread to complete
    thread.join()

Started thread but waiting for event...
Event is set.Event received, releasing thread...

